# Krones — Submission notebook (ResNet-50 + YOLO ROI + TTA)

Pipeline:
1. YOLO ONNX predicts ROI bbox on each test image
2. Expand ROI by 150 px (matches training crops)
3. Resize crop to 512×512
4. Run classifier with 8-way TTA (4 rotations × 2 flips), average probabilities
5. Apply threshold (set from `infer_tta.py` on val), write 0/1 to submission.csv

## Required Kaggle dataset uploads
Add as datasets in this notebook:
- `roi-onnx`: directory containing `best.onnx` (your YOLO ROI model)
- `krones-classifier`: directory containing `best.pt` (the trained ResNet-50)

Then update the paths in the **Paths** cell below to match.

## Notes
- Set `THRESHOLD` to whatever `infer_tta.py --split val` printed.
- ImageNet normalization stats hard-coded; must match training.


In [ ]:
!pip uninstall -y onnxruntime-gpu onnxruntime

In [ ]:
# Setup
import os
import gc
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision.models import resnet50
from torchvision import transforms
import onnxruntime as ort
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# Paths - adjust to match your Kaggle dataset names
COMP_DIR = Path("/home/dll0202/baksh/dataset/1st-krones-vision-ai-challenge")
TEST_IMG_DIR = COMP_DIR / "test_images"
SAMPLE_SUB = COMP_DIR / "sample_submission.csv"

# adjust these two to where you uploaded the model files
YOLO_ONNX = Path("/home/dll0202/baksh/krones_2.0/output/ROI_Detector/best.onnx")
CLF_CKPT = Path(
    "/home/dll0202/baksh/krones_2.0/Codebase/Codebase_2.0/runs/resnet50_640/best.pt"
)

# Inference config
ROI_PAD = 150  # MUST match training crop padding
IMG_SIZE = 512  # MUST match training img-size
BATCH_SIZE = 16
NUM_WORKERS = 4
THRESHOLD = 0.445  # <-- set this from infer_tta.py output on val
YOLO_INPUT = 640  # YOLO was exported at imgsz=640

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

for p in (TEST_IMG_DIR, SAMPLE_SUB, YOLO_ONNX, CLF_CKPT):
    print(f"{p}  exists: {p.exists()}")
print(f"\nTHRESHOLD = {THRESHOLD}")

In [ ]:
# ---- Stage 1: YOLO ROI detection on every test image ----
# Loads ONNX, runs inference batch-by-batch, returns dict file_name -> [x, y, w, h]

providers = (
    ["CUDAExecutionProvider", "CPUExecutionProvider"]
    if device == "cuda"
    else ["CPUExecutionProvider"]
)
yolo_sess = ort.InferenceSession(str(YOLO_ONNX), providers=providers)
yolo_input_name = yolo_sess.get_inputs()[0].name
yolo_output_names = [o.name for o in yolo_sess.get_outputs()]
print("YOLO providers:", yolo_sess.get_providers())
print("YOLO input:", yolo_input_name, yolo_sess.get_inputs()[0].shape)
print("YOLO outputs:", yolo_output_names)


def preprocess_for_yolo(img_pil):
    """Resize to YOLO_INPUT, return tensor (1,3,H,W) float32 [0,1]."""
    img = img_pil.convert("RGB").resize((YOLO_INPUT, YOLO_INPUT))
    arr = np.asarray(img, dtype=np.float32) / 255.0
    arr = arr.transpose(2, 0, 1)[None]
    return arr


def parse_yolo_output(out, orig_w, orig_h):
    """YOLOv8/YOLO26 export returns (1, N, 6) where columns are [x1,y1,x2,y2,score,class] in YOLO_INPUT space.
    Pick the highest-scoring detection (single ROI class) and rescale to original image coords.
    Returns [x, y, w, h] in original-image pixels, or None if no detection."""
    arr = out
    if arr.ndim == 3:
        arr = arr[0]  # (N, 6)
    if arr.shape[0] == 0:
        return None
    # pick top-confidence row
    if arr.shape[1] >= 6:
        scores = arr[:, 4]
    else:
        return None
    idx = int(np.argmax(scores))
    if scores[idx] < 0.1:
        return None
    x1, y1, x2, y2 = arr[idx, :4]
    sx = orig_w / YOLO_INPUT
    sy = orig_h / YOLO_INPUT
    x1 *= sx
    x2 *= sx
    y1 *= sy
    y2 *= sy
    return [float(x1), float(y1), float(x2 - x1), float(y2 - y1)]


# Read submission template to get test order
sub = pd.read_csv(SAMPLE_SUB)
print(f"test images to process: {len(sub)}")

# Run YOLO on each test image, save its ROI bbox
roi_by_fn = {}
missing = 0
fallback = 0
t0 = time.time()
for i, fn in enumerate(tqdm(sub["image_id"].tolist(), desc="YOLO ROI")):
    img_path = TEST_IMG_DIR / fn
    if not img_path.exists():
        missing += 1
        continue
    with Image.open(img_path) as im:
        W, H = im.size
        x = preprocess_for_yolo(im)
    out = yolo_sess.run(yolo_output_names, {yolo_input_name: x})[0]
    bbox = parse_yolo_output(out, W, H)
    if bbox is None:
        # fallback: full image
        bbox = [0, 0, W, H]
        fallback += 1
    roi_by_fn[fn] = (bbox, W, H)

print(f"\nROI prediction done in {time.time() - t0:.1f}s")
print(f"  missing files: {missing}")
print(f"  fallback (no detection, used full image): {fallback}")
del yolo_sess
gc.collect()

In [ ]:
# ---- Stage 2: classifier model + dataset ----

ckpt = torch.load(CLF_CKPT, map_location=device, weights_only=False)
print(
    f"classifier ckpt: epoch {ckpt['epoch']}  trained val F1 {ckpt.get('val_f1', float('nan')):.4f}"
)

model = resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 1)
model.load_state_dict(ckpt["model"])
model = model.to(device).eval()

tfm = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)


class KronesTestDataset(Dataset):
    def __init__(self, sub_df, roi_by_fn, img_dir):
        self.df = sub_df
        self.roi_by_fn = roi_by_fn
        self.img_dir = img_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        fn = self.df.iloc[i]["image_id"]
        img_path = self.img_dir / fn
        bbox, W, H = self.roi_by_fn.get(fn, ([0, 0, 0, 0], 1, 1))
        x, y, w, h = bbox
        x0 = max(0, int(x) - ROI_PAD)
        y0 = max(0, int(y) - ROI_PAD)
        x1 = min(W, int(x + w) + ROI_PAD)
        y1 = min(H, int(y + h) + ROI_PAD)
        with Image.open(img_path) as img:
            crop = img.crop((x0, y0, x1, y1)).convert("RGB")
        return tfm(crop), fn


ds = KronesTestDataset(sub, roi_by_fn, TEST_IMG_DIR)
loader = DataLoader(
    ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True
)
print(f"test loader: {len(ds)} samples, {len(loader)} batches")

In [ ]:
# ---- Stage 3: TTA inference (4 rotations x 2 flips = 8 passes) ----


@torch.no_grad()
def tta_predict(model, imgs, autocast_dtype):
    parts = []
    for k in range(4):
        x = torch.rot90(imgs, k, dims=[-2, -1])
        with torch.amp.autocast("cuda", dtype=autocast_dtype):
            p = torch.sigmoid(model(x).squeeze(1).float())
        parts.append(p)
        xf = torch.flip(x, dims=[-1])
        with torch.amp.autocast("cuda", dtype=autocast_dtype):
            pf = torch.sigmoid(model(xf).squeeze(1).float())
        parts.append(pf)
    return torch.stack(parts, dim=0).mean(dim=0)


all_probs = np.zeros(len(ds), dtype=np.float32)
all_fns = [None] * len(ds)
i = 0
t0 = time.time()
for imgs, fns in tqdm(loader, desc="TTA"):
    imgs = imgs.to(device, non_blocking=True)
    probs = tta_predict(model, imgs, torch.bfloat16).cpu().numpy()
    n = len(fns)
    all_probs[i : i + n] = probs
    all_fns[i : i + n] = list(fns)
    i += n
print(f"TTA done in {time.time() - t0:.1f}s")

In [ ]:
# ---- Stage 4: write submission ----
preds = (all_probs >= THRESHOLD).astype(int)
out_df = pd.DataFrame({"image_id": all_fns, "target": preds})
# match the row order of sample_submission.csv exactly
out_df = sub[["image_id"]].merge(out_df, on="image_id", how="left")
assert out_df["target"].notna().all(), "some predictions missing"
out_df["target"] = out_df["target"].astype(int)

out_path = "/home/dll0202/baksh/krones_2.0/Codebase/Codebase_2.0/runs/baseline_resnet50/submission.csv"
out_df.to_csv(out_path, index=False)
print(f"wrote {out_path}")
print(out_df["target"].value_counts())
print("\npreview:")
print(out_df.head(10).to_string(index=False))

In [ ]:
# Sanity check: probability histogram
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 3))
plt.hist(all_probs, bins=50)
plt.axvline(THRESHOLD, color="red", linestyle="--", label=f"thr={THRESHOLD}")
plt.xlabel("predicted Faulty probability")
plt.ylabel("count")
plt.title("TTA probability distribution on test set")
plt.legend()
plt.tight_layout()
plt.show()

# rough: how many items are 'borderline' (prob 0.4-0.6)?
borderline = ((all_probs >= 0.4) & (all_probs <= 0.6)).sum()
print(
    f"items with prob in [0.4, 0.6]: {borderline} ({borderline / len(all_probs) * 100:.2f}%)"
)